# Distributed Data Parallel (DDP) — training across GPUs

`torch.distributed` is PyTorch's built-in module for multi-GPU and multi-node training.
This is NOT a third-party library — it ships with PyTorch.

## Why DDP exists

Single GPU can't train large models:
- LLaMA-7B needs ~28GB just for weights in float32
- Plus gradients (28GB) + optimizer states (56GB for AdamW) = ~112GB
- Best consumer GPU (RTX 4090) has 24GB

**Data Parallelism:** copy the model to N GPUs, split the batch, each GPU processes its chunk, then sync gradients.

## DDP vs DataParallel

| | `nn.DataParallel` | `nn.parallel.DistributedDataParallel` |
|---|---|---|
| How | Single process, GIL-limited | One process PER GPU |
| Speed | Slow (GIL bottleneck) | Fast (no GIL, overlapped comms) |
| Use | Never in production | Always use this |

## Java analogy
DataParallel = single-threaded with synchronized blocks.
DDP = separate JVM processes communicating via message passing (like Akka/MPI).

## Important
DDP requires multiple GPUs or process simulation. This notebook teaches concepts + code patterns.
Run the actual multi-GPU code on Colab (multi-GPU runtime) or simulate with CPU processes.

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
print(f'GPU count:       {torch.cuda.device_count() if torch.cuda.is_available() else 0}')
print(f'NCCL available:  {torch.distributed.is_nccl_available()}')

## How DDP works — the algorithm

```
1. Each GPU ("rank") has a FULL COPY of the model
2. Data is split: each rank gets different mini-batch
3. Each rank does forward + backward independently
4. Gradients are ALL-REDUCED (averaged across ranks)
5. Each rank does optimizer.step() with the SAME averaged gradients
6. Result: all model copies stay identical

   Rank 0              Rank 1              Rank 2
  ┌─────────┐       ┌─────────┐       ┌─────────┐
  │ Model   │       │ Model   │       │ Model   │
  │ (copy)  │       │ (copy)  │       │ (copy)  │
  └────┬────┘       └────┬────┘       └────┬────┘
       │                 │                 │
  Batch chunk 0    Batch chunk 1    Batch chunk 2
       │                 │                 │
  Forward + Loss   Forward + Loss   Forward + Loss
       │                 │                 │
  Backward         Backward         Backward
       │                 │                 │
       └─────── ALL-REDUCE gradients ──────┘
                (average gradients)
       │                 │                 │
  optimizer.step() optimizer.step() optimizer.step()
  (same update)    (same update)    (same update)
```

**Key insight:** the model sees `N * batch_size` examples per step but each GPU only does `batch_size` work.

In [ ]:
# --- DDP Training Script (pattern you'll use in real projects) ---
# This is written as a function because DDP needs to spawn separate processes

def ddp_train(rank, world_size):
    """
    rank:       this GPU's ID (0, 1, 2, ...)
    world_size: total number of GPUs
    """
    # 1. Initialize process group
    dist.init_process_group(
        backend='gloo',           # 'nccl' for GPU, 'gloo' for CPU simulation
        init_method='tcp://localhost:12355',
        rank=rank,
        world_size=world_size
    )

    # 2. Create model and wrap with DDP
    model = nn.Sequential(
        nn.Linear(100, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )
    # On GPU: model = model.to(rank)
    # On GPU: model = DDP(model, device_ids=[rank])
    model = DDP(model)  # CPU mode for simulation

    # 3. DistributedSampler ensures each rank gets different data
    dataset = TensorDataset(
        torch.randn(1000, 100),
        torch.randint(0, 10, (1000,))
    )
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader  = DataLoader(dataset, batch_size=32, sampler=sampler)

    # 4. Standard training loop — DDP handles gradient sync automatically
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(3):
        sampler.set_epoch(epoch)   # IMPORTANT: ensures different shuffling per epoch
        total_loss = 0.0

        for x, y in loader:
            logits = model(x)
            loss   = nn.functional.cross_entropy(logits, y)

            optimizer.zero_grad()
            loss.backward()        # DDP automatically all-reduces gradients here
            optimizer.step()

            total_loss += loss.item()

        if rank == 0:  # only print from rank 0 (avoid duplicate logs)
            print(f'Epoch {epoch} | Loss: {total_loss / len(loader):.4f}')

    # 5. Cleanup
    dist.destroy_process_group()


print('DDP training function defined.')
print('In real usage: mp.spawn(ddp_train, args=(world_size,), nprocs=world_size)')
print('This spawns one process per GPU, each running ddp_train independently.')

## Running DDP

```python
# Option 1: mp.spawn (from within Python)
if __name__ == '__main__':
    world_size = torch.cuda.device_count()
    mp.spawn(ddp_train, args=(world_size,), nprocs=world_size)

# Option 2: torchrun (from command line — preferred in production)
# $ torchrun --nproc_per_node=4 train.py
# torchrun sets RANK, WORLD_SIZE, LOCAL_RANK automatically
```

**Production always uses `torchrun`** because:
- Handles node failures and restarts
- Sets environment variables correctly
- Supports multi-node across machines

In [ ]:
# --- Simulate all-reduce on CPU to understand the algorithm ---

# Pretend 4 GPUs each computed different gradients
gpu_0_grad = torch.tensor([1.0, 2.0, 3.0])
gpu_1_grad = torch.tensor([4.0, 5.0, 6.0])
gpu_2_grad = torch.tensor([7.0, 8.0, 9.0])
gpu_3_grad = torch.tensor([2.0, 3.0, 1.0])

all_grads = torch.stack([gpu_0_grad, gpu_1_grad, gpu_2_grad, gpu_3_grad])

# All-reduce = average across all GPUs
averaged = all_grads.mean(dim=0)

print('Individual gradients per GPU:')
for i, g in enumerate(all_grads):
    print(f'  GPU {i}: {g.tolist()}')

print(f'\nAll-reduced (averaged): {averaged.tolist()}')
print('\nAfter all-reduce, EVERY GPU uses this same averaged gradient.')
print('That is why model copies stay in sync.')

## DDP checklist (common mistakes)

| Mistake | Fix |
|---|---|
| Forgot `sampler.set_epoch(epoch)` | Data not reshuffled → overfitting |
| Logging from all ranks | Duplicate prints — only log from `rank == 0` |
| Saving model from all ranks | Save only from `rank == 0` |
| `model.save()` instead of `model.module.save()` | DDP wraps your model — `.module` gets the unwrapped one |
| Using `nn.DataParallel` | Always use DDP, never DataParallel |
| Not calling `dist.barrier()` before loading | Race condition on shared filesystem |

## What's beyond DDP — covered in systems-for-ml/

DDP copies the full model to every GPU. What if the model doesn't fit on one GPU?

| Strategy | What it does | Tool |
|---|---|---|
| **ZeRO Stage 1** | Partition optimizer states | DeepSpeed |
| **ZeRO Stage 2** | + partition gradients | DeepSpeed |
| **ZeRO Stage 3** | + partition model weights | DeepSpeed |
| **Model parallelism** | Split layers across GPUs | Megatron-LM |
| **Pipeline parallelism** | Split model into stages | DeepSpeed, GPipe |

These are third-party frameworks built ON TOP of PyTorch — covered in `systems-for-ml/`.